# OpenAI Responses API

## What is the OpenAI Responses API?

The Responses API is a new API released in March 2025. It is a combination of the traditional 
Chat Completions API and the Assistants API, providing support for:

- **Traditional Chat Completions:** Facilitates seamless conversational AI experiences.
- **Web Search:** Enables real-time information retrieval from the internet.
- **File Search:** Allows searching within files for relevant data.

Accordingly, the Assistants API will be retired in 2026. 

> **For new users, OpenAI recommends using the Responses API instead of the Chat Completions API to leverage its expanded capabilities.**

For a comprehensive comparison between the Responses API and the Chat Completions API, refer to the official OpenAI documentation: 
[Responses vs. Chat Completions](https://platform.openai.com/docs/guides/responses-vs-chat-completions).

## Summary of This Notebook
This notebook provides a hands-on guide for using the **OpenAI Responses API** to analyze tweets. 
It covers essential techniques such as:

- **Creating a vector store** and uploading tweets for semantic search.
- **Using file search** to analyze private datasets.
- **Performing a web search** to retrieve the latest public information.
- **Utilizing stateful responses** to maintain conversation context.
- **Combining file and web search** to enhance retrieval-augmented generation (RAG) applications.

By the end of this notebook, users will be able to integrate OpenAI's Responses API for efficient data retrieval and analysis of structured and unstructured data.

## Install Required Libraries
To use the OpenAI Responses API, we need to install the following libraries:

- **`openai`**: Provides access to OpenAI's APIs, including the Responses API

In [1]:
pip install openai -q

Note: you may need to restart the kernel to use updated packages.


## Import Required Libraries

In [2]:
from IPython.display import Markdown, display
import boto3
from botocore.exceptions import ClientError
import json
import io

## Retrieve Secrets from AWS Secrets Manager

In [3]:
def get_secret(secret_name):
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        raise e

    secret = get_secret_value_response['SecretString']
    
    return json.loads(secret)

## Initialize OpenAI Client

In [4]:
from openai import OpenAI
openai_api_key  = get_secret('openai')['api_key']

client = OpenAI(api_key=openai_api_key)

## File Search API

### Introduction to File Search
File search API enables efficient retrieval of relevant information 
from uploaded files by leveraging vector-based indexing. This feature is particularly useful 
for searching large datasets, extracting insights, and improving retrieval-augmented generation (RAG) applications.

Unlike traditional keyword-based searches, the Responses API uses embeddings 
to identify semantically relevant content, making it ideal for analyzing structured 
and unstructured text data (OpenAI, 2025).

For more details, visit the official OpenAI documentation: 
[File Search in Responses API](https://platform.openai.com/docs/guides/tools-file-search).

### Create a Vector Store

In [5]:
vector_store = client.vector_stores.create(
    name="my_vector_store"
)
vector_store_id = vector_store.id
print(vector_store_id)

vs_6912648066148191bce0bda0288be737


### Upload Files

In [7]:
with open('tweet_text (3).json', 'rb') as f:
    file = client.files.create(
        file=f,            # file-like object
        purpose="assistants"
    )

file_id = file.id
print(file_id)

file-QtJF76rrd5Trbt47Fgnrgi


### Attach File to Vector Store

In [8]:
attach_status =client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_id
            )

print(attach_status.id)

file-QtJF76rrd5Trbt47Fgnrgi


### Query the Vector Store

In [9]:
query = "the latest development in generativeAI"

In [10]:
search_results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=query
)

for result in search_results.data[:5]:
    print(result.content[0].text[:100] + '\n Relevant score: ' + str(result.score))

They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1f1
 Relevant score: 0.6067149500128853
Learning Plans. Use coupon code 𝐒𝐏𝐋𝟑𝟎 at checkout.🎯\n\nJoin Now 👉 https://t.co/LS2JuCrVmz\n\n#AI #Ce
 Relevant score: 0.6008365042590297
Create stunning, cinematic videos from a prompt—now with audio, physics, and cameos. One prompt = en
 Relevant score: 0.5950557796727322
They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1d1
 Relevant score: 0.5624679641016309
🤤 AI is making cinematic food commercials in minutes. The sizzling patty, the dripping cheese... no 
 Relevant score: 0.550412199769088


## OpenAI Response API

### Simple Response

In [11]:
simple_response = client.responses.create(
  model="gpt-4o",
  input=[
      {
          "role": "user",
          "content": query
      }
  ]
)

In [12]:
display(Markdown(simple_response.output_text))

As of the latest updates, generative AI continues to make significant strides across various domains:

1. **Multimodal Models**: Models that can process and generate text, images, and even audio or video are becoming more advanced. These models can understand and create complex content across different media types, enhancing applications like content creation and virtual reality.

2. **Efficient Scaling**: Researchers are focusing on making large language models more efficient, reducing resource requirements while maintaining or improving performance. Techniques like parameter-efficient fine-tuning help make models more accessible and environmentally friendly.

3. **Ethical and Responsible AI**: There's increasing emphasis on ensuring AI models are developed and deployed responsibly. This includes addressing biases, improving transparency, and creating guidelines for ethical AI use.

4. **Enhanced Creativity Tools**: AI is being integrated into tools for creative professionals, offering features like automated design suggestions, music composition, and story generation, allowing for collaboration between humans and AI.

5. **Domain-Specific Generative Models**: Developments in creating models tailored for specific domains such as medicine, law, and finance help provide more relevant and precise insights and solutions.

6. **Open-Source Initiatives**: More companies and organizations are releasing open-source generative models, allowing broader access to powerful tools and fostering innovation across the community.

These advancements reflect a growing trend towards more versatile, accessible, and ethically designed AI systems that can assist in a wide range of applications.

### File Search Response

In [13]:

file_search_response = client.responses.create(
    input= query,
    model="gpt-4o",
    temperature = 0,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    }]
)

In [14]:
display(Markdown(file_search_response.output_text))


The latest developments in generative AI include several exciting advancements:

1. **OpenAI's Sora2**: This tool allows users to create cinematic videos from a prompt, now with added features like audio, physics, and cameos.

2. **Generative AI in Business**: It's reshaping innovation across sectors, from content generation to design systems.

3. **AI in Supply Chain**: Atos has developed an AI-powered Supply Chain Disruption Analysis using generative AI to assess risk and boost resilience.

4. **Market Growth**: The global generative AI market is expected to reach $1.18 billion this year.

5. **Enterprise Tools**: IBM's Watsonx is bringing generative AI to enterprises with tools for secure, responsible deployment.

These developments highlight the expanding role of generative AI in various industries, enhancing creativity, efficiency, and innovation.

## Web Search API

### Introduction to Web Search
The OpenAI Web Search tool allows models to retrieve real-time information from the internet. 
This capability is particularly useful for obtaining up-to-date data, fact-checking, and expanding knowledge 
without relying solely on pre-trained information. 

By leveraging OpenAI's web search functionality, the Responses API can fetch external data 
and provide accurate, relevant results in real time (OpenAI, 2025). 
This feature enhances applications that require the latest insights, such as news aggregation, research, 
or dynamic content generation.

For more details, visit the official OpenAI documentation: 
[Web Search in Responses API](https://platform.openai.com/docs/guides/tools-web-search).

### Perform Web Search

In [15]:
web_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [16]:
display(Markdown(web_search_response.output_text))

Here’s an updated and comprehensive overview of the **latest developments in generative AI** as of November 10, 2025. This includes recent model releases, hardware innovations, open-source progress, and emerging trends in AI autonomy and regulation.

---

##  Recent Model Releases & Enhancements

**1. OpenAI’s GPT‑5 (launched August 7, 2025)**  
- GPT‑5 is a powerful multimodal large language model combining advanced reasoning with large-scale multimodal capabilities (text, image, voice) under a unified interface. It powers ChatGPT and Microsoft Copilot and is accessible via OpenAI’s API. ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5?utm_source=openai))  
- New "light" variants — GPT‑5-mini and GPT‑5-nano — optimized for smartphones and edge devices, broaden the model’s practical use beyond the cloud. ([scrumlaunch.com](https://www.scrumlaunch.com/blog/top-5-breakthrough-ai-launches-summer-2025?utm_source=openai))  
- GPT‑5 offers a massive 256,000-token context window, robust “thinking mode” for planning, abstraction, and error correction, and outperforms GPT-4 by over 40% in challenging logical reasoning benchmarks. ([scrumlaunch.com](https://www.scrumlaunch.com/blog/top-5-breakthrough-ai-launches-summer-2025?utm_source=openai))

**2. OpenAI’s o4‑mini (released April 16, 2025)**  
- A reasoning-focused transformer, o4‑mini processes both text and images and can analyze whiteboard sketches during internal reasoning steps. It’s available to all ChatGPT users, with the high-accuracy “o4‑mini‑high” reserved for paid tiers. ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenAI_o4-mini?utm_source=openai))

**3. Anthropic’s Claude 3.7 Sonnet (released February 24, 2025)**  
- A hybrid reasoning model with two modes: fast answers or extended “thinking mode” that self-reflects before responding. It improves reasoning across math, physics, and coding domains. ([reuters.com](https://www.reuters.com/technology/artificial-intelligence/anthropic-launches-advanced-ai-hybrid-reasoning-model-2025-02-24/?utm_source=openai))  
- Includes Claude Code, a coding assistant that can autonomously browse codebases, write tests, and integrate with GitHub from the terminal. ([reuters.com](https://www.reuters.com/technology/artificial-intelligence/anthropic-launches-advanced-ai-hybrid-reasoning-model-2025-02-24/?utm_source=openai))

**4. Google DeepMind’s Gemini Series**  
- Gemini 2.5 Pro (released in early 2025) is Google’s most intelligent model yet, featuring enhanced reasoning, coding, and multimodal inputs with a 1-million-token context window and Deep Think mode. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Gemini_%28language_model%29?utm_source=openai))  
- In August 2025, Google launched **“Nano Banana”** (Gemini 2.5 Flash Image), a viral AI image-generation editor with photorealistic 3D figurine outputs, multi-image fusion, and SynthID watermarking. It attracted over 10 million new users and exceeded 200 million image edits within weeks. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Nano_Banana?utm_source=openai))

**5. Meta’s Llama 4 Family (early 2025)**  
- Meta introduced Llama 4’s Scout (109B parameters, 16 experts) and Maverick (402B parameters, 128 experts) using a mixture-of-experts (MoE) architecture. These multimodal, multilingual models support massive 10M-token context windows while optimizing compute cost. A larger “Behemoth” model (2T parameters) is in training. ([ts2.tech](https://ts2.tech/en/generative-ai-revolution-2025-breakthroughs-industry-disruption-and-predictions-through-2035/?utm_source=openai))

---

##  Hardware & Infrastructure Developments

**6. Nvidia’s Next-Gen AI Chips (announced March 18, 2025)**  
- Nvidia introduced **Blackwell Ultra** and the future **Rubin AI chip**, aiming for launches in late 2026 and 2027, respectively. These architectures support reasoning and generative/agentic AI, underpinning robotic and physical-AI systems. ([apnews.com](https://apnews.com/article/457e9260aa2a34c1bbcc07c98b7a0555?utm_source=openai))  
- The company also unveiled **Isaac GR00T N1**, an open-source humanoid robotics model, and **Cosmos AI** for synthetic training data, alongside a physics engine (Newton) for robotics simulation. ([apnews.com](https://apnews.com/article/457e9260aa2a34c1bbcc07c98b7a0555?utm_source=openai))

**7. Qualcomm’s AI Center Infrastructure (announced October 2025)**  
- Qualcomm revealed AI200 and AI250 NPUs, high-efficiency inference accelerators targeting 2026–2027 deployment. These support Gen AI encryption, micro-tile inferencing, and scalable rack-based architectures, competing with AMD and Nvidia in data centers. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/qualcomm-unveils-ai200-and-ai250-ai-inference-accelerators-hexagon-takes-on-amd-and-nvidia-in-the-booming-data-center-realm?utm_source=openai))

---

##  Open-Source & Democratization

**8. OpenAI’s gpt‑oss-120b and gpt‑oss-20b (released August 2025)**  
- Open-weight, open-access models supporting chain-of-thought reasoning. The 20B model runs locally on consumer hardware like Snapdragon PCs, while the 120B variant requires more powerful GPUs. Broad availability via Hugging Face, Azure, AWS, and other platforms. ([windowscentral.com](https://www.windowscentral.com/artificial-intelligence/openai-chatgpt/openai-launches-two-gpt-models-theyre-not-gpt-5-but-they-run-locally-on-snapdragon-pcs-and-nvidia-rtx-gpus?utm_source=openai))

---

##  Broader Themes & Trends

**9. Growing Agentic AI**  
- Autonomous AI agents in 2025 can independently strategize, negotiate, and execute complex tasks, ranging from marketing campaigns to contract negotiation — dynamically adapting to local feedback. ([vocal.media](https://vocal.media/education/5-major-generative-ai-breakthroughs-that-are-shaping-2025?utm_source=openai))  
- Claude Code (Anthropic) exemplifies this trend with autonomous code generation and deployment. ([reuters.com](https://www.reuters.com/technology/artificial-intelligence/anthropic-launches-advanced-ai-hybrid-reasoning-model-2025-02-24/?utm_source=openai))

**10. Efficiency, Accessibility & Privacy**  
- AI models have become more resource-efficient, reducing compute costs and expanding access for smaller organizations. ([techinnoai.com](https://techinnoai.com/artificial-intelligence-machine-learning/generative-ai-in-2025-breakthroughs-real-world-applications/?utm_source=openai))  
- On-device generative AI is rising—for instance, Apple is embedding privacy-first AI tools (like Genmoji and Visual Intelligence) directly on iPhones and Macs. ([launchconsulting.com](https://www.launchconsulting.com/posts/may-2025-ai-breakthroughs-what-every-business-leader-needs-to-know?utm_source=openai))

**11. Regulation & Ethical AI**  
- Countries including the U.S., EU, Canada, and Japan are establishing frameworks for AI transparency, risk assessment, and ethical deployment. The EU AI Act is already in force. ([ferip.com](https://ferip.com/5-powerful-breakthroughs-the-future-of-ai-in-2025/?utm_source=openai))

**12. Academic Perspectives**  
- New semantic information-theoretic frameworks are emerging to improve generative AI’s fidelity and meaning in multimedia communication. ([arxiv.org](https://arxiv.org/abs/2508.17163?utm_source=openai))  
- Interactive generative video models, conceptualized as generative game engines, promise dynamic, physics-aware environments for future gaming. ([arxiv.org](https://arxiv.org/abs/2503.17359?utm_source=openai))  
- Research at CHI 2025 explores how generative AI tools can augment and protect human cognition—spanning metacognition, creativity, and ethical design. ([arxiv.org](https://arxiv.org/abs/2508.21036?utm_source=openai))

---

## Summary Table (Late 2025 Highlights)

| Area              | Key Developments |
|-------------------|------------------|
| **Model Innovation** | GPT‑5, o4‑mini, Claude 3.7, Gemini 2.5 Pro, Nano Banana, Llama 4 |
| **Hardware**         | Nvidia’s Rubin architecture, Qualcomm’s AI200/AI250 |
| **Open Source**      | gpt-oss models for local deployment |
| **Trends**           | Agentic AI, efficiency, on-device AI, AI regulation |
| **Academic Research**| Semantically richer generative systems, cognition-focused design |

---

**In conclusion**, the latest developments in generative AI (as of November 10, 2025) reveal a fast-evolving landscape marked by more powerful, multimodal models, sophisticated hardware support, open-source democratization, ethical governance, and groundbreaking applications in agentic systems, human augmentation, and interactive media. Let me know if you'd like to dive deeper into any of these specific areas!

### Stateful Response

The OpenAI Responses API includes a stateful feature that enables continuity in interactions. 
By using the `response_id`, a conversation can persist across multiple queries, 
allowing users to refine or expand upon previous searches. This is particularly useful for iterative research, 
dynamic content generation, and applications that require follow-up queries based on prior responses.

In [17]:
fetched_response = client.responses.retrieve(response_id=web_search_response.id)
display(Markdown(fetched_response.output_text[:100]))

Here’s an updated and comprehensive overview of the **latest developments in generative AI** as of N

### Continue Query with Web Search

In [18]:
continue_query = 'find different news'

continue_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= continue_query,
    previous_response_id=web_search_response.id,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [19]:
display(Markdown(continue_search_response.output_text))

Here are several **recent and diverse news developments in generative AI**, providing a well-rounded snapshot of innovations across industries, applications, and infrastructure as of November 10, 2025. Each item includes multiple citations for credibility.

---

###  1. Semiconductor Boom Driven by Generative AI Demand  
The rise of generative AI models is fueling an unprecedented demand for advanced computing infrastructure, ushering in what experts are calling a **“silicon supercycle.”** This surge is reshaping the global semiconductor industry, positioning AI data centers as the new growth engine for chipmakers.  
([markets.financialcontent.com](https://markets.financialcontent.com/wral/article/tokenring-2025-11-10-the-silicon-supercycle-how-ai-data-centers-are-forging-a-new-era-for-semiconductors?utm_source=openai))

---

###  2. Insilico Medicine’s AI-Driven Drug Discovery Portfolio  
On November 7, 2025, **Insilico Medicine** announced a breakthrough in drug discovery using generative AI. They unveiled a **cardiometabolic disease portfolio**—a set of highly differentiated molecular assets—designed using AI-driven methods. This demonstrates how AI is accelerating innovation in pharmaceutical R&D.  
([eurekalert.org](https://www.eurekalert.org/news-releases/1105110?utm_source=openai))

---

###  3. Adobe’s Generative AI Expands Creative Toolset  
At **Adobe MAX 2025**, Adobe introduced expanded generative capabilities across its creative suite, particularly through its **Firefly** platform. New features include:
- **Generate Soundtrack** and **Generate Speech**, allowing auto-creation of music and voiceovers with emotional and stylistic guidance.
- **Layered image editing** in Firefly Image Model 5, enabling manipulation of discrete elements without artifacts.
- A browser-based multi-track video editor integrating video, audio, and images.
- **Project Moonlight**, linking user context across creative apps with a private beta launched at the event.  
([wired.com](https://www.wired.com/story/adobe-max-2025-firefly-photoshop-updates?utm_source=openai))

Additionally, Adobe introduced **AI Foundry**, enabling businesses to train brand-specific generative models using proprietary media assets. Early adopters include Home Depot and Walt Disney Imagineering. Following this announcement, Adobe’s stock rose around 2.5%, although the company remains under pressure in its sector.  
([investors.com](https://www.investors.com/news/technology/adobe-stock-adbe-tailored-gen-ai-models/?utm_source=openai))

---

###  4. TIME Launches the “TIME AI Agent” for Interactive Journalism  
TIME magazine introduced the **TIME AI Agent**, a unified AI-powered platform developed in collaboration with Scale AI. This tool enhances how readers interact with journalism by combining:
- Language understanding
- Voice synthesis
- Translation
- Search

It enables personalized content experiences such as generating summaries, creating audio reports, translating articles, and exploring news interactively—while upholding editorial standards.  
([time.com](https://time.com/7332572/the-story-behind-the-time-ai-agent/?utm_source=openai))

---

###  5. Sydney Informatics Hub Empowers Australian Researchers with GenAI Tools  
The University of Sydney’s **Sydney Informatics Hub** is actively helping Australia’s research community adopt generative AI responsibly. The initiative supports researchers across disciplines in navigating both opportunities and challenges posed by these powerful tools.  
([sydney.edu.au](https://www.sydney.edu.au/news-opinion/news/2025/11/04/sydney-informatics-hub-unlocks-generative-ai-for-research.html?utm_source=openai))

---

###  6. Summit Launches ‘AI and Robotics Fest’ in Hong Kong  
On November 10, 2025, the **GBA International Artificial Intelligence and Robotics Summit** kicked off in Hong Kong. The event—dubbed “AI and Robotics Fest”—was launched by the Hong Kong Productivity Council to drive the adoption of AI and embodied AI (robotics), signaling a major push to transform productivity and innovation in the region.  
([manilatimes.net](https://www.manilatimes.net/2025/11/10/tmt-newswire/media-outreach-newswire/gba-international-artificial-intelligence-and-robotics-summit-2025-opens-grandly-ai-and-robotics-fest-launches-alongside-hkpc-drives-ai-for-all-and-embodied-ai-adoption-to-propel-hong-kong-into-a-new-era-of-new-productive-forces/2219964?utm_source=openai))

---

**Summary of Key Themes:**
- Generative AI is driving **hardware demand** and reshaping the **semiconductor industry**.
- Applications span across **healthcare (drug discovery)**, **creative industries (Adobe tools)**, **media (TIME’s journalism platform)**, **academic research**, and **industrial robotics**.
- There’s a clear trend toward integrating generative AI into both **professional workflows** and **public-facing platforms**, while adhering to responsible design and governance.

Let me know if you'd like to explore any of these news items in greater depth!

### Combining File Search and Web Search

This is an example of using file search to analyze private data and web search to retrieve public or the latest data. 
The Responses API allows developers to integrate these tools to enhance retrieval-augmented generation (RAG) applications. 
By combining file search with web search, users can leverage structured internal knowledge while also retrieving real-time 
information from external sources, ensuring comprehensive and up-to-date responses. 

In [20]:
combined_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    temperature = 0,
    instructions="Retrieve the results from the file search first, and use the web search tool to expand the results with news resources",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    },
        {
            "type": "web_search"
        }
    ]
)

In [21]:
display(Markdown(combined_search_response.output_text))

Here’s a comprehensive and up-to-date overview of the **latest developments in generative AI** as of November 10, 2025. This analysis draws on recent news, model releases, market trends, and emerging applications.

---

##  Major Model Releases and Innovations

- **OpenAI’s GPT‑5**  
  Released on **August 7, 2025**, GPT‑5 is a multimodal foundation model that unifies reasoning and non-reasoning capabilities under a single interface. It is accessible via ChatGPT, Microsoft Copilot, and the OpenAI API, and represents a significant leap in performance across benchmarks. ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5?utm_source=openai))

- **OpenAI’s GPT‑4.1 and o4‑mini**  
  - **GPT‑4.1** launched on **April 14, 2025**, offering improved coding and reasoning capabilities. It includes variants like GPT‑4.1 mini and nano, with broader availability across ChatGPT Plus and Pro tiers. ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-4.1?utm_source=openai))  
  - **o4‑mini**, released on **April 16, 2025**, is a lightweight multimodal model capable of processing text and images, including whiteboard sketches, and is available to all ChatGPT users. ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenAI_o4-mini?utm_source=openai))

- **Google DeepMind’s Gemini Series and “Nano Banana”**  
  - The **Gemini 2.5** family, including Flash and Pro variants, introduced advanced reasoning, coding, and “Deep Think” capabilities, with general availability announced on **June 17, 2025**. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Gemini_%28language_model%29?utm_source=openai))  
  - **Nano Banana** (Gemini 2.5 Flash Image), launched on **August 26, 2025**, is a viral image generation and editing model known for photorealistic “3D figurine” outputs, multi-image fusion, and SynthID watermarking. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Nano_Banana?utm_source=openai))

- **Google’s Gemini Diffusion**  
  A novel experimental model that applies diffusion techniques to text generation, enabling simultaneous generation of text segments with mid-process corrections. It achieves speeds up to **1,479 tokens per second**, significantly faster than traditional LLMs. ([spglobal.com](https://www.spglobal.com/market-intelligence/en/news-insights/research/generative-ai-digest-a-wave-of-notable-ai-model-launches?utm_source=openai))

- **Other Notable Models**  
  - **Anthropic’s Claude Haiku 4.5**, a compact model delivering flagship-level performance with a 200K token context window, high speed, and low cost. ([voxfor.com](https://www.voxfor.com/what-is-new-in-ai-the-latest-news-from-october-2025/?utm_source=openai))  
  - **Alibaba’s Qwen3** family, released in April 2025, includes open-source models (0.6B to 32B parameters) with “thinking” and “nonthinking” modes, optimized for coding, math, and instruction tasks. ([spglobal.com](https://www.spglobal.com/market-intelligence/en/news-insights/research/generative-ai-digest-a-wave-of-notable-ai-model-launches?utm_source=openai))

---

##  Market Trends and Enterprise Adoption

- **Explosive Market Growth**  
  The generative AI market experienced triple-digit growth across hardware, foundation models, and development platforms in 2024. Forecasts estimate over **US$400 billion in AI-related spending in 2025**. ([businesswire.com](https://www.businesswire.com/news/home/20250825682581/en/Generative-AI-Market-Report-2025-GenAI-Market-Experienced-Triple-digit-growth-Rates-in-All-Three-Major-Segments-Spanning-GenAI-Hardware-Foundation-Models-and-Development-Platforms---ResearchAndMarkets.com?utm_source=openai))

- **Enterprise Scaling and Agentic AI**  
  According to the **2025 McKinsey Global Survey on AI**, about **23% of organizations are scaling agentic AI systems**, while **39% are experimenting** with them. High-performing organizations are more likely to redesign workflows, invest over 20% of digital budgets in AI, and embed human validation processes. ([mckinsey.com](https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai?utm_source=openai))

- **Generative AI in Software Development**  
  A **Bain report (September 23, 2025)** highlights that while generative AI tools boost productivity, realizing real value requires process changes and thoughtful integration. ([bain.com](https://www.bain.com/insights/from-pilots-to-payoff-generative-ai-in-software-development-technology-report-2025/?utm_source=openai))

---

##  Responsible Use and Governance

- **UC Berkeley’s Responsible GenAI Playbook**  
  Released in mid-2025, this playbook outlines **10 actionable strategies**—five for business leaders and five for product managers—to ensure responsible development and deployment of generative AI. ([weforum.org](https://www.weforum.org/stories/2025/06/responsible-generative-ai-product-development-use/?utm_source=openai))

---

##  Emerging Applications and Ecosystem Integration

- **TIME AI Agent**  
  Launched **today (November 10, 2025)**, the TIME AI Agent is a generative AI platform that enhances reader interaction with journalism. It offers summaries, audio reports, translations, and interactive exploration, built in partnership with Scale AI and grounded in editorial standards. ([time.com](https://time.com/7332572/the-story-behind-the-time-ai-agent/?utm_source=openai))

- **Microsoft’s MAI‑Image‑1**  
  Released **5 days ago**, this proprietary text-to-image model is integrated into Bing Image Creator and Copilot Audio Expressions. It offers fast, photorealistic image generation and complements OpenAI models within Microsoft’s ecosystem. ([windowscentral.com](https://www.windowscentral.com/artificial-intelligence/microsoft-copilot/microsoft-launches-mai-image-1?utm_source=openai))

- **Weekly AI Highlights (Nov 1–7, 2025)**  
  - **Sora for Android**: OpenAI’s video app achieved **470,000 downloads on its first day**. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  - **Google’s DS STAR**: A multi-agent framework that converts ambiguous business problems into executable Python code. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  - **Meta’s Vibes**: AI-generated short video platform expanded to Europe. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  - **Google Maps + Gemini**: Integration of Gemini as a voice assistant for navigation. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  - **HeyGen’s AI Video Translator**: Offers hyper-realistic localization with matching tone and lip movements. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  - **Gemini 3 Pro Preview**: Spotted on Vertex AI, expected to support a **1-million-token context window**. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  - **ClickUp 4.0**: Introduced AI agents and a redesigned UI for unified task management. ([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))

---

##  Summary

Generative AI in late 2025 is characterized by:

- **Advanced multimodal models** like GPT‑5, Gemini 2.5, and Nano Banana.
- **Rapid market expansion**, with enterprises scaling agentic AI and investing heavily.
- **Focus on responsible AI**, with frameworks guiding ethical deployment.
- **Deep integration into products**, from journalism to navigation and productivity tools.

Let me know if you'd like a deeper dive into any specific model, application, or trend!

# 🧩 Try It Yourself: Two-Step RAG (Private Data + Combined Search)

## Step 1 — Upload & Create Vector Store
1. Upload a short text file (e.g., `my_notes.txt`) to your notebook instance.  
2. Create a **vector store** and **ingest** your uploaded file.  
3. Run a simple test query to verify retrieval:  

In [23]:
tiy_vector_store = client.vector_stores.create(
    name="tiy_vector_store"
)
tiy_vector_store_id = tiy_vector_store.id
print("Vector store created:", tiy_vector_store_id)

with open("343Lab10NewsClip.txt", "rb") as f:
    uploaded_file = client.files.create(
        file=f,
        purpose="assistants"
    )

print("File uploaded:", uploaded_file.id)


attach_result = client.vector_stores.files.create(
    vector_store_id=tiy_vector_store_id,
    file_id=uploaded_file.id,
)
print("File attached to vector store:", attach_result.id)


test_query = "Summarize the main ideas from my notes."
search_results = client.vector_stores.search(
    vector_store_id=tiy_vector_store_id,
    query=test_query,
)

print("\nTop retrieved chunks:")
for item in search_results.data[:3]:
    print("-" * 40)
    print(item.content[0].text.strip())

vs_6912683ddd6081918b29bf74a3135d97
file-RxJTLzfYoVtfuy8npbz5Jg


## Step 2 — Combine File Search with Web Search
1. Enable both **file_search** and **web_search** in the Responses API.  
2. Use a prompt that asks the model to merge insights from both sources.  
   > Example: “Using my uploaded notes and the latest web information, summarize the current trends on this topic.”  
3. Review how the answer from your file and **current info** from the web.

✅ You’ve created a RAG system that combines **private** and **public** data for comprehensive, up-to-date analysis.


In [33]:
query = (
    "Using the uploaded news article as the primary source, and also checking the latest web information, "
    "give me a short, structured summary of the topic (El Paso Texas and change following immigration crackdown). Clearly separate 'From my file' vs 'From the web'."
)

combined_response = client.responses.create(
    model="gpt-4o",
    input=query,
    temperature=0,
    instructions=(
        "First look up relevant passages from the attached vector store. "
        "Then augment with web_search to bring in current/public info. "
        "Present the answer in two sections: 'From my file(s)' and 'From the web'."
    ),
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [tiy_vector_store_id],
        },
        {
            "type": "web_search"
        }
    ],
)

display(Markdown(combined_response.output_text))

NameError: name 'tiy_vector_store_id' is not defined